# Distributed Evolutionary Algorithms
**Author:** @VedantAndhale

In [7]:
import random
from deap import base, creator, tools, algorithms

## 1. Define the Problem

We will optimize a simple function: minimize the sum of squares of a vector. This is a standard test function for evolutionary algorithms.

In [8]:
def eval_func(individual):
    """Objective: Minimize the sum of squares."""
    return (sum(x**2 for x in individual),)

## 2. DEAP Setup

We define the fitness, individual, and toolbox for the evolutionary algorithm. The individual is a list of floats, and the fitness is minimized.

In [9]:
# DEAP setup
if "FitnessMin" not in creator.__dict__:
    creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
if "Individual" not in creator.__dict__:
    creator.create("Individual", list, fitness=creator.FitnessMin)

toolbox = base.Toolbox()

# Define attributes and individuals
toolbox.register(
    "attr_float", random.uniform, -5.0, 5.0
)  # Example: Float values between -5 and 5
toolbox.register(
    "individual", tools.initRepeat, creator.Individual, toolbox.attr_float, n=3
)  # Example: 3-dimensional individual
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

## 3. Register Genetic Operators

We register the evaluation, crossover, mutation, and selection operators. These are the core components of the genetic algorithm.

In [10]:
# Evaluation function and genetic operators
toolbox.register("evaluate", eval_func)
toolbox.register("mate", tools.cxBlend, alpha=0.5)
toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=1, indpb=0.2)
toolbox.register("select", tools.selTournament, tournsize=3)

## 4. Hybridization: Integrating Local Search

To create a hybrid algorithm, we add a local search step. After each generation, we apply a simple hill climbing to the best individual to further refine the solution.

In [11]:
def hill_climb(individual, eval_func, steps=10, step_size=0.1):
    """Simple hill climbing: tries small changes to improve the individual."""
    best = list(individual)
    best_score = eval_func(best)[0]
    for _ in range(steps):
        candidate = [x + random.uniform(-step_size, step_size) for x in best]
        score = eval_func(candidate)[0]
        if score < best_score:
            best = candidate
            best_score = score
    return creator.Individual(best)

## 5. Run the Hybrid Evolutionary Algorithm

We run the genetic algorithm as usual, but after each generation, we apply hill climbing to the best individual and replace it in the population if it improves.

In [12]:
# Create population
population = toolbox.population(n=50)

generations = 20

for gen in range(generations):
    offspring = algorithms.varAnd(population, toolbox, cxpb=0.5, mutpb=0.1)

    fits = toolbox.map(toolbox.evaluate, offspring)
    for fit, ind in zip(fits, offspring):
        ind.fitness.values = fit

    # Hybrid step: apply hill climbing to the best individual
    best = tools.selBest(offspring, k=1)[0]
    improved = hill_climb(best, eval_func)
    improved.fitness.values = eval_func(improved)
    if improved.fitness.values[0] < best.fitness.values[0]:
        # Replace the worst individual with the improved one
        worst = tools.selWorst(offspring, k=1)[0]
        offspring[offspring.index(worst)] = improved

    population = toolbox.select(offspring, k=len(population))

## 6. Results and Discussion

We print the best solution found and discuss how hybridization can help evolutionary algorithms escape local optima and converge faster.

In [13]:
best_ind = tools.selBest(population, k=1)[0]
best_fitness = best_ind.fitness.values[0]

print("Best individual:", best_ind)
print("Best fitness:", best_fitness)

Best individual: [-0.0015939302936478252, -0.001856617732198326, 0.0003191028400088023]
Best fitness: 6.08946980702318e-06
